In [1]:
# ============================================================
# LEVEL 3: RAG + EMBEDDINGS + CHROMADB + GEMINI
# No LangChain
# No LlamaIndex
# Google Colab - Single Cell
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL LIBRARIES
# ------------------------------------------------------------

!pip install -q chromadb sentence-transformers google-genai

In [2]:
# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import userdata

In [3]:
# ============================================================
# 3. GEMINI SETUP
# ============================================================

# In Google Colab:
# Left panel -> Secrets
# Add:
#
# GEMINI_API_KEY

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [4]:
# ============================================================
# 4. LOAD EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
print("Embedding model loaded.")

# ============================================================
# 5. SAMPLE KNOWLEDGE BASE
# ============================================================

documents = [

    """
    Retrieval-Augmented Generation, or RAG, combines
    information retrieval with a large language model.
    The system first retrieves relevant information and
    then supplies that information to the LLM as context.
    """,

    """
    ChromaDB is a vector database used to store embeddings.
    It supports similarity search and is commonly used
    in Retrieval-Augmented Generation applications.
    """,

    """
    Embeddings are numerical vector representations of data.
    Semantically similar pieces of text usually have vectors
    that are close to each other in vector space.
    """,

    """
    Gemini is Google's family of large multimodal models.
    Gemini can generate text, analyze information and answer
    questions based on context supplied in a prompt.
    """,

    """
    Vector similarity search compares an input query vector
    with stored document vectors. Common similarity measures
    include cosine similarity, dot product and Euclidean distance.
    """,

    """
    In a RAG system, documents are normally divided into smaller
    chunks before creating embeddings. This improves retrieval
    because the system can retrieve only the most relevant
    sections instead of entire documents.
    """
]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [5]:
# ============================================================
# 6. DOCUMENT IDS
# ============================================================

ids = [
    "doc1",
    "doc2",
    "doc3",
    "doc4",
    "doc5",
    "doc6"
]


# ============================================================
# 7. CREATE DOCUMENT EMBEDDINGS
# ============================================================

print("\nCreating document embeddings...")
document_embeddings = embedding_model.encode(
    documents
).tolist()
print(
    "Number of documents:",
    len(documents)
)
print(
    "Embedding dimensions:",
    len(document_embeddings[0])
)

# ============================================================
# 8. CREATE CHROMADB CLIENT
# ============================================================

chroma_client = chromadb.Client()

# ============================================================
# 9. CREATE COLLECTION
# ============================================================

# Delete old collection if it exists

try:
    chroma_client.delete_collection(
        name="rag_knowledge_base"
    )
except:
    pass
collection = chroma_client.create_collection(
    name="rag_knowledge_base"
)


Creating document embeddings...
Number of documents: 6
Embedding dimensions: 384


In [8]:
# ============================================================
# 10. STORE DOCUMENTS + EMBEDDINGS IN CHROMADB
# ============================================================

collection.add(
    ids=ids,
    documents=documents,
    embeddings=document_embeddings
)

print(
    "\nDocuments stored in ChromaDB."
)
print(
    "Total records:",
    collection.count()
)


# ============================================================
# 11. ASK USER A QUESTION
# ============================================================

question = input(
    "\nEnter your question: "
)

# ============================================================
# 12. CONVERT QUESTION INTO EMBEDDING
# ============================================================

query_embedding = embedding_model.encode(
    question
).tolist()

print(
    "\nQuestion converted into vector embedding."
)

# ============================================================
# 13. SEARCH CHROMADB
# ============================================================

results = collection.query(
    query_embeddings=[
        query_embedding
    ],
    n_results=3
)


# ============================================================
# 14. GET TOP MATCHING DOCUMENTS
# ============================================================

retrieved_documents = (
    results["documents"][0]
)
retrieved_distances = (
    results["distances"][0]
)


# ============================================================
# 15. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print(
    "\n" + "=" * 60
)
print(
    "RETRIEVED DOCUMENTS"
)
print(
    "=" * 60
)

for i, (
    document,
    distance
) in enumerate(

    zip(
        retrieved_documents,
        retrieved_distances
    ),
    start=1

):

    print(
        f"\nResult {i}"
    )
    print(
        "Distance:",
        round(distance, 4)
    )
    print(
        document.strip()
    )


# ============================================================
# 16. BUILD CONTEXT
# ============================================================

context = "\n\n".join(
    retrieved_documents
)


# ============================================================
# 17. CREATE RAG PROMPT
# ============================================================

prompt = f"""
You are an AI assistant using Retrieval-Augmented Generation.

Answer the user's question using ONLY the information
provided in the retrieved context below.

If the answer is not available in the context, say:

"I do not have enough information in the knowledge base."

Do not invent information.

========================

RETRIEVED CONTEXT:
{context}

========================

USER QUESTION:
{question}

========================

ANSWER:
"""

# ============================================================
# 18. SEND CONTEXT + QUESTION TO GEMINI
# ============================================================

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# ============================================================
# 19. DISPLAY FINAL ANSWER
# ============================================================

print(
    "\n" + "=" * 60
)
print(
    "FINAL RAG ANSWER"
)
print(
    "=" * 60
)
print()
print(
    response.text
)


Documents stored in ChromaDB.
Total records: 6

Enter your question: What is RAG?

Question converted into vector embedding.

RETRIEVED DOCUMENTS

Result 1
Distance: 0.8966
In a RAG system, documents are normally divided into smaller
    chunks before creating embeddings. This improves retrieval
    because the system can retrieve only the most relevant
    sections instead of entire documents.

Result 2
Distance: 1.0713
Retrieval-Augmented Generation, or RAG, combines
    information retrieval with a large language model.
    The system first retrieves relevant information and
    then supplies that information to the LLM as context.

Result 3
Distance: 1.7372
ChromaDB is a vector database used to store embeddings.
    It supports similarity search and is commonly used
    in Retrieval-Augmented Generation applications.

FINAL RAG ANSWER

RAG, or Retrieval-Augmented Generation, combines information retrieval with a large language model. The system first retrieves relevant information